# Compute mean change confidence intervals

Full temporal coverage: compute NDVI change confidence intervals for all same-month consecutive-year pairs from 2019 to 2025 across all study areas. Results and configuration are saved under `outputs/change_ci/all_year_pairs_2019_2025`.

In [1]:
from pathlib import Path

import pandas as pd

from change_ci import AnalysisConfig, ChangeCIAnalysis, build_pairs, discover_rasters

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"

## Analysis configuration

In [2]:
config = AnalysisConfig.from_yaml(
    ROOT / "configs" / "change_ci.yaml", project_root=ROOT
)
config.to_dict()

{'data_root': '/home/tshihadah/code/wadi-gaza-research/outputs/composites',
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/all_year_pairs_2019_2025',
 'study_areas': None,
 'months': ('01', '05', '09', '11'),
 'year_range': (2019, 2025),
 'index_band': 'NDVI',
 'engine': 'fft',
 'tau': 0.05,
 'tau_sensitivity': (0.02, 0.05, 0.1),
 'alpha': 0.05,
 'max_lag_fraction': 0.25,
 'min_n_eff': 30}

## Discover all same-month consecutive-year pairs

Expected coverage: 168 pairs (seven study areas × four months × six year transitions).

In [3]:
raster_records = discover_rasters(config.data_root)
pair_specs = build_pairs(raster_records, config)
pair_table = pd.DataFrame([
    {
        'study_area': pair.study_area,
        'month': pair.month,
        'year_start': pair.year_start,
        'year_end': pair.year_end,
        'earlier_path': pair.earlier_path,
        'later_path': pair.later_path,
    }
    for pair in pair_specs
])
pair_table

,study_area,month,year_start,year_end,earlier_path,later_path
0,AlAhrashProtectorate,01,2019,2020,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
1,AlAhrashProtectorate,01,2020,2021,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
2,AlAhrashProtectorate,01,2021,2022,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
3,AlAhrashProtectorate,01,2022,2023,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
4,AlAhrashProtectorate,01,2023,2024,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
...,...,...,...,...,...,...
163,WadiGaza,11,2020,2021,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
164,WadiGaza,11,2021,2022,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
165,WadiGaza,11,2022,2023,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
166,WadiGaza,11,2023,2024,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...


## Run and save

In [4]:
analysis = ChangeCIAnalysis(config)
results, correlograms = analysis.run(pair_specs)
analysis.save(results, correlograms)
print(f'Saved {len(results)} result rows and {len(correlograms)} correlogram rows to {OUTPUT_DIR}')

Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


... (159 outputs left out)

Raster pixels are not square: (10.000130544932714, 10.006954791496103)


Raster pixels are not square: (10.000130544932714, 10.006954791496103)


Raster pixels are not square: (10.000130544932714, 10.006954791496103)


Saved 504 result rows and 12744 correlogram rows to /home/tshihadah/code/wadi-gaza-research/outputs/change_ci/all_year_pairs_2019_2025


## Results and review flags

In [5]:
primary = results.loc[results['is_primary_tau']].copy()
summary_columns = [
    'study_area', 'month', 'year_start', 'year_end', 'n', 'mean_change', 'sd',
    'range_px', 'range_map_units', 'n_eff', 'ci_low', 'ci_high',
    'naive_ci_low', 'naive_ci_high', 'flag_no_range', 'flag_low_n_eff', 'error',
]
display(primary[summary_columns].sort_values('study_area'))

review = primary.loc[
    primary['flag_no_range']
    | primary['flag_low_n_eff']
    | primary['error'].fillna('').ne('')
]
print(f'{len(review)} primary-threshold result(s) flagged or failed.')
display(review[summary_columns].sort_values('study_area'))

,study_area,month,year_start,year_end,n,mean_change,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
1,AlAhrashProtectorate,01,2019,2020,33848,0.025359,0.047606,26.0,259.983591,97.081448,0.015769,0.034950,0.024852,0.025867,False,False,
70,AlAhrashProtectorate,11,2024,2025,33848,0.021026,0.024055,NaN,NaN,NaN,NaN,NaN,0.020769,0.021282,True,False,
67,AlAhrashProtectorate,11,2023,2024,33848,-0.038127,0.052321,NaN,NaN,NaN,NaN,NaN,-0.038685,-0.037570,True,False,
64,AlAhrashProtectorate,11,2022,2023,33848,0.023829,0.040096,44.0,439.972231,38.422989,0.010727,0.036930,0.023402,0.024256,False,False,
61,AlAhrashProtectorate,11,2021,2022,33848,0.013320,0.027641,45.0,449.971599,47.134039,0.005217,0.021424,0.013026,0.013615,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,WadiGaza,01,2020,2021,28412,0.080548,0.184842,16.0,160.002089,231.416583,0.056607,0.104489,0.078399,0.082697,False,False,
433,WadiGaza,01,2019,2020,28413,-0.062779,0.179344,11.0,110.001436,361.635704,-0.081326,-0.044233,-0.064865,-0.060694,False,False,
499,WadiGaza,11,2023,2024,28413,-0.153810,0.190717,33.0,330.004308,79.568708,-0.196370,-0.111249,-0.156027,-0.151592,False,False,
463,WadiGaza,05,2023,2024,28413,-0.098711,0.144143,27.0,270.003525,126.182137,-0.124107,-0.073316,-0.100387,-0.097035,False,False,


25 primary-threshold result(s) flagged or failed.


,study_area,month,year_start,year_end,n,mean_change,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
25,AlAhrashProtectorate,05,2021,2022,33848,0.018449,0.025881,NaN,NaN,NaN,NaN,NaN,0.018173,0.018725,True,False,
31,AlAhrashProtectorate,05,2023,2024,33848,-0.027111,0.042746,NaN,NaN,NaN,NaN,NaN,-0.027566,-0.026655,True,False,
34,AlAhrashProtectorate,05,2024,2025,33848,0.006739,0.020266,NaN,NaN,NaN,NaN,NaN,0.006523,0.006955,True,False,
49,AlAhrashProtectorate,09,2023,2024,33848,-0.016620,0.035279,NaN,NaN,NaN,NaN,NaN,-0.016996,-0.016244,True,False,
52,AlAhrashProtectorate,09,2024,2025,33848,0.005293,0.019672,NaN,NaN,NaN,NaN,NaN,0.005084,0.005503,True,False,
67,AlAhrashProtectorate,11,2023,2024,33848,-0.038127,0.052321,NaN,NaN,NaN,NaN,NaN,-0.038685,-0.037570,True,False,
70,AlAhrashProtectorate,11,2024,2025,33848,0.021026,0.024055,NaN,NaN,NaN,NaN,NaN,0.020769,0.021282,True,False,
79,BesorStream,01,2021,2022,11318,-0.058620,0.105673,NaN,NaN,NaN,NaN,NaN,-0.060567,-0.056673,True,False,
82,BesorStream,01,2022,2023,11318,-0.036762,0.099973,NaN,NaN,NaN,NaN,NaN,-0.038604,-0.034920,True,False,
88,BesorStream,01,2024,2025,11318,-0.265510,0.134059,NaN,NaN,NaN,NaN,NaN,-0.267980,-0.263040,True,False,
